#### fichier deprecated permetant d'obtenir la première base de données de ML for climate risk 

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

from fonction import *

 ### preprocessing donnée communes

In [2]:
from pathlib import Path

fp = Path("data") / "raw" / "communes" / "ADE_4-0_GPKG_WGS84G_FRA-ED2025-11-20.gpkg"
if not fp.exists():
	raise FileNotFoundError(f"File not found: {fp.resolve()}")
communes = gpd.read_file(fp, layer='commune')

In [ ]:
communes

,cleabs,nom_officiel,nom_officiel_en_majuscules,statut,code_insee,population,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region,codes_siren_des_epci,code_siren,code_postal,superficie_cadastrale,geometry
0,COMMUNE_0000000000001001,L'Abergement-Clémenciat,L'ABERGEMENT-CLEMENCIAT,Commune simple,01001,859,2022-01-01,INSEE,0108,012,01,84,200069193,210100012,01400,1590,"MULTIPOLYGON (((4.95841 46.15327, 4.95812 46.1..."
1,COMMUNE_0000000000001002,L'Abergement-de-Varey,L'ABERGEMENT-DE-VAREY,Commune simple,01002,273,2022-01-01,INSEE,0101,011,01,84,240100883,210100020,01640,920,"MULTIPOLYGON (((5.4302 45.98277, 5.43012 45.98..."
2,COMMUNE_0000000000001004,Ambérieu-en-Bugey,AMBERIEU-EN-BUGEY,Commune simple,01004,15554,2022-01-01,INSEE,0101,011,01,84,240100883,210100046,01500,2460,"MULTIPOLYGON (((5.40882 45.94206, 5.4085 45.94..."
3,COMMUNE_0000000000001005,Ambérieux-en-Dombes,AMBERIEUX-EN-DOMBES,Commune simple,01005,1917,2022-01-01,INSEE,0122,012,01,84,200042497,210100053,01330,1590,"MULTIPOLYGON (((4.94298 45.97962, 4.94257 45.9..."
4,COMMUNE_0000000000001006,Ambléon,AMBLEON,Commune simple,01006,114,2022-01-01,INSEE,0104,011,01,84,200040350,210100061,01300,590,"MULTIPOLYGON (((5.57083 45.75338, 5.57219 45.7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34872,COMMUNE_0000000000097611,Mamoudzou,MAMOUDZOU,Préfecture de région,97611,71437,2017-01-01,INSEE,97699,NR,976,06,200060457,200008837,PLURI,4230,"MULTIPOLYGON (((45.23784 -12.81009, 45.23791 -..."
34873,COMMUNE_0000000000097614,Ouangani,OUANGANI,Commune simple,97614,10203,2017-01-01,INSEE,97610,NR,976,06,200059871,200008852,97670,1850,"MULTIPOLYGON (((45.15409 -12.87424, 45.14544 -..."
34874,COMMUNE_0000000000097607,Dembeni,DEMBENI,Commune simple,97607,15848,2017-01-01,INSEE,97603,NR,976,06,200060457,200008787,97660,3838,"MULTIPOLYGON (((45.16269 -12.87999, 45.15888 -..."
34875,COMMUNE_0000000000097613,M'Tsangamouji,M'TSANGAMOUJI,Commune simple,97613,6432,2017-01-01,INSEE,97613,NR,976,06,200059871,200008829,97650,2204,"MULTIPOLYGON (((45.06646 -12.75976, 45.06631 -..."


In [4]:
# Liste des codes INSEE des départements d'outre-mer
outre_mer_codes = ['971', '972', '973', '974', '976']

# Filtrer les communes pour exclure celles ayant un code INSEE d'outre-mer
communes = communes[~communes['code_insee_du_departement'].isin(outre_mer_codes)]

In [8]:
# Chemin vers le fichier GeoPackage
gpkg_path = Path("data/intermediate/datagouv/datagouv_swimonthly_2010to2019.gpkg")

#monthly_df_to_gpkg(datagouv_monthly, gpkg_path,crs="EPSG:27572")

In [15]:

# Define file paths
csv_file_path = "data/raw/datagouv/QUOT_SIM2_1958-1959.csv"
parquet_file_path = "data/intermediate/datagouv/QUOT_SIM2_1958-1959.parquet"

csv_to_parquet(csv_file_path, parquet_file_path, sep=';')


datagouv=pd.read_parquet(parquet_file_path, engine='pyarrow')
### transformation de la date en format datetime
datagouv['DATE'] = pd.to_datetime(datagouv['DATE'], format='%Y%m%d')


# Agrégation des données au mois pour chaque paire de coordonnées (LAMBX, LAMBY) en utilisant un traitement par chunks
chunk_size = 10_000_000  # Nombre de lignes à traiter par chunk
chunks = []

for start in range(0, len(datagouv), chunk_size):
    end = start + chunk_size
    chunk = datagouv.iloc[start:end]
    
    aggregated_chunk = chunk.groupby(
        [pd.Grouper(key='DATE', freq='ME'), 'LAMBX', 'LAMBY']
    ).agg({
        'PRENEI': 'sum',       # Somme des précipitations solides
        'PRELIQ': 'sum',       # Somme des précipitations liquides
        'T': 'mean',           # Moyenne des températures
        'FF': 'mean',          # Moyenne du vent
        'Q': 'mean',           # Moyenne de l'humidité spécifique
        'DLI': 'sum',          # Somme du rayonnement atmosphérique
        'SSI': 'sum',          # Somme du rayonnement visible
        'HU': 'mean',          # Moyenne de l'humidité relative
        'EVAP': 'sum',         # Somme de l'évapotranspiration totale
        'ETP': 'sum',          # Somme de l'évapotranspiration potentielle
        'PE': 'sum',           # Somme des pluies efficaces
        'SWI': 'mean',         # Moyenne de l'indice d'humidité des sols
        'SSWI_10J': 'mean',    # Moyenne de l'indice de sécheresse des sols
        'DRAINC': 'sum',       # Somme du drainage
        'RUNC': 'sum',         # Somme du ruissellement
        'RESR_NEIGE': 'mean',  # Moyenne de l'équivalent en eau du manteau neigeux
        'RESR_NEIGE6': 'mean', # Moyenne de l'équivalent en eau du manteau neigeux à 6h
        'HTEURNEIGE': 'mean',  # Moyenne de l'épaisseur du manteau neigeux
        'HTEURNEIGE6': 'mean', # Moyenne de l'épaisseur du manteau neigeux à 6h
        'HTEURNEIGEX': 'max',  # Maximum de l'épaisseur horaire du manteau neigeux
        'SNOW_FRAC': 'mean',   # Moyenne de la fraction de maille recouverte par la neige
        'ECOULEMENT': 'sum',   # Somme de l'écoulement
        'WG_RACINE': 'mean',   # Moyenne du contenu en eau liquide dans la couche racinaire
        'WGI_RACINE': 'mean',  # Moyenne du contenu en eau gelée dans la couche racinaire
        'TINF_H': 'min',       # Température minimale
        'TSUP_H': 'max',       # Température maximale
    }).reset_index()
    
    chunks.append(aggregated_chunk)

# Combiner les résultats des chunks
datagouv_monthly = pd.concat(chunks, ignore_index=True)

# Afficher un aperçu des données agrégées
display(datagouv_monthly)

# Chemin vers le fichier GeoPackage
gpkg_path = Path("data/intermediate/datagouv/datagouv_swimonthly_1958-1959.gpkg")

monthly_df_to_gpkg(datagouv_monthly, gpkg_path,units='hm', crs="EPSG:27572")


Le fichier Parquet existe déjà : data\intermediate\datagouv\QUOT_SIM2_1958-1959.parquet


,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,RESR_NEIGE6,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H
0,1958-08-31,600,24010,0.0,101.1,16.183871,3.735484,10.297871,104077.3,31484.2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.234903,0.0,10.3,21.4
1,1958-08-31,760,23610,0.0,85.0,16.277419,4.748387,10.441677,103165.5,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.174645,0.0,11.5,21.5
2,1958-08-31,760,23930,0.0,84.1,16.400000,4.719355,10.511032,103705.2,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.226355,0.0,11.6,21.8
3,1958-08-31,760,24010,0.0,103.3,15.819355,3.800000,10.094516,102541.1,31493.2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.220677,0.0,10.2,20.6
4,1958-08-31,760,24090,0.0,102.4,15.958065,3.764516,10.168677,103106.1,31489.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.217355,0.0,10.2,20.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168159,1959-12-31,11960,17050,0.0,63.7,10.774194,3.664516,6.051387,85616.0,14624.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.298968,0.0,3.7,20.0
168160,1959-12-31,11960,17130,0.0,152.4,10.751613,3.522581,6.045258,87003.8,13470.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.344645,0.0,4.8,19.8
168161,1959-12-31,11960,17210,0.0,152.5,10.735484,3.522581,6.041419,86961.7,13470.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.332903,0.0,4.8,19.7
168162,1959-12-31,11960,17290,0.0,152.5,10.735484,3.522581,6.041419,86961.7,13470.6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.349677,0.0,4.8,19.7


Le fichier GPKG existe déjà : C:\Users\thoma\OneDrive\Bureau\boulot\climate risk\Climate_Risk_Project_SWI_MS_A\data\intermediate\datagouv\datagouv_swimonthly_1958-1959.gpkg


WindowsPath('data/intermediate/datagouv/datagouv_swimonthly_1958-1959.gpkg')

In [16]:


# Read the GeoPackage file
datagouv_previous = gpd.read_file(gpkg_path)

# Display the first few rows of the data
display(datagouv_previous)

,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry
0,1958-08-31,600,24010,0.0,101.1,16.183871,3.735484,10.297871,104077.3,31484.2,...,0.0,0.0,0.0,0.0,0.0,0.234903,0.0,10.3,21.4,POINT (60000 2401000)
1,1958-08-31,760,23610,0.0,85.0,16.277419,4.748387,10.441677,103165.5,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.174645,0.0,11.5,21.5,POINT (76000 2361000)
2,1958-08-31,760,23930,0.0,84.1,16.400000,4.719355,10.511032,103705.2,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.226355,0.0,11.6,21.8,POINT (76000 2393000)
3,1958-08-31,760,24010,0.0,103.3,15.819355,3.800000,10.094516,102541.1,31493.2,...,0.0,0.0,0.0,0.0,0.0,0.220677,0.0,10.2,20.6,POINT (76000 2401000)
4,1958-08-31,760,24090,0.0,102.4,15.958065,3.764516,10.168677,103106.1,31489.5,...,0.0,0.0,0.0,0.0,0.0,0.217355,0.0,10.2,20.9,POINT (76000 2409000)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168159,1959-12-31,11960,17050,0.0,63.7,10.774194,3.664516,6.051387,85616.0,14624.0,...,0.0,0.0,0.0,0.0,0.0,0.298968,0.0,3.7,20.0,POINT (1196000 1705000)
168160,1959-12-31,11960,17130,0.0,152.4,10.751613,3.522581,6.045258,87003.8,13470.4,...,0.0,0.0,0.0,0.0,0.0,0.344645,0.0,4.8,19.8,POINT (1196000 1713000)
168161,1959-12-31,11960,17210,0.0,152.5,10.735484,3.522581,6.041419,86961.7,13470.6,...,0.0,0.0,0.0,0.0,0.0,0.332903,0.0,4.8,19.7,POINT (1196000 1721000)
168162,1959-12-31,11960,17290,0.0,152.5,10.735484,3.522581,6.041419,86961.7,13470.6,...,0.0,0.0,0.0,0.0,0.0,0.349677,0.0,4.8,19.7,POINT (1196000 1729000)


In [17]:


# Répertoire contenant les fichiers GeoPackage swimonthly
swimonthly_dir = Path("data/intermediate/datagouv")
output_parquet = swimonthly_dir / "swimonthly_combined_1958to2025.parquet"
output_gpkg = swimonthly_dir / "swimonthly_combined_1958to2025.gpkg"

# Liste des fichiers GeoPackage swimonthly
swimonthly_files = list(swimonthly_dir.glob("datagouv_swimonthly_*.gpkg"))

# Vérifier s'il y a des fichiers GeoPackage
if not swimonthly_files:
    raise FileNotFoundError(f"Aucun fichier GeoPackage swimonthly trouvé dans le répertoire {swimonthly_dir.resolve()}")

# Lire et concaténer tous les fichiers GeoPackage
combined_gdf = gpd.GeoDataFrame(pd.concat([gpd.read_file(file) for file in swimonthly_files], ignore_index=True))

display(combined_gdf)

,DATE,LAMBX,LAMBY,PRENEI,PRELIQ,T,FF,Q,DLI,SSI,...,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry
0,1958-08-31,600,24010,0.0,101.1,16.183871,3.735484,10.297871,104077.3,31484.2,...,0.0,0.0,0.0,0.0,0.0,0.234903,0.0,10.3,21.4,POINT (60000 2401000)
1,1958-08-31,760,23610,0.0,85.0,16.277419,4.748387,10.441677,103165.5,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.174645,0.0,11.5,21.5,POINT (76000 2361000)
2,1958-08-31,760,23930,0.0,84.1,16.400000,4.719355,10.511032,103705.2,31824.6,...,0.0,0.0,0.0,0.0,0.0,0.226355,0.0,11.6,21.8,POINT (76000 2393000)
3,1958-08-31,760,24010,0.0,103.3,15.819355,3.800000,10.094516,102541.1,31493.2,...,0.0,0.0,0.0,0.0,0.0,0.220677,0.0,10.2,20.6,POINT (76000 2401000)
4,1958-08-31,760,24090,0.0,102.4,15.958065,3.764516,10.168677,103106.1,31489.5,...,0.0,0.0,0.0,0.0,0.0,0.217355,0.0,10.2,20.9,POINT (76000 2409000)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7982859,2025-10-31,11960,17050,0.0,44.5,18.122581,2.538710,8.597774,91740.2,33351.6,...,0.0,0.0,0.0,0.0,0.0,0.190774,0.0,12.5,25.8,POINT (1196000 1705000)
7982860,2025-10-31,11960,17130,0.0,27.8,16.987097,2.683871,7.945290,92372.0,33203.6,...,0.0,0.0,0.0,0.0,0.0,0.194065,0.0,9.3,25.8,POINT (1196000 1713000)
7982861,2025-10-31,11960,17210,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.0,0.0,0.0,0.0,0.0,0.186806,0.0,9.3,25.8,POINT (1196000 1721000)
7982862,2025-10-31,11960,17290,0.0,27.8,16.967742,2.683871,7.937194,92353.8,33201.1,...,0.0,0.0,0.0,0.0,0.0,0.204161,0.0,9.3,25.8,POINT (1196000 1729000)


In [18]:
# Sauvegarder en format Parquet
combined_gdf.to_parquet(output_parquet, index=False)
print(f"Fichier Parquet combiné sauvegardé à {output_parquet.resolve()}")

# Sauvegarder en format GeoPackage
combined_gdf.to_file(output_gpkg, driver="GPKG")
print(f"Fichier GeoPackage combiné sauvegardé à {output_gpkg.resolve()}")

Fichier Parquet combiné sauvegardé à C:\Users\thoma\OneDrive\Bureau\boulot\climate risk\Climate_Risk_Project_SWI_MS_A\data\intermediate\datagouv\swimonthly_combined_1958to2025.parquet
Fichier GeoPackage combiné sauvegardé à C:\Users\thoma\OneDrive\Bureau\boulot\climate risk\Climate_Risk_Project_SWI_MS_A\data\intermediate\datagouv\swimonthly_combined_1958to2025.gpkg


| Nom du champ | Descriptif                                                                                                                     | Unité   | Précision |
|--------------|-------------------------------------------------------------------------------------------------------------------------------|---------|-----------|
| DATE         | Date de la mesure                                                                                                             | AAAAMMJJ|           |
| PRENEI       | Précipitations solides, cumul quotidien ]06UTC-06UTC]                                                                        | mm      | 1/10      |
| PRELIQ       | Précipitations liquides, cumul quotidien ]06UTC-06UTC]                                                                       | mm      | 1/10      |
| T            | Température moyenne quotidienne ]00UTC-00UTC]                                                                                | °C      | 1/10      |
| FF           | Vent moyen quotidien ]00UTC-00UTC]                                                                                           | m/s     | 1/10      |
| Q            | Humidité spécifique moyenne quotidienne ]00UTC-00UTC]                                                                        | g/kg    |           |
| DLI          | Rayonnement atmosphérique, cumul quotidien ]00UTC-00UTC]                                                                     | J/cm²   |           |
| SSI          | Rayonnement visible, cumul quotidien ]00UTC-00UTC]                                                                           | J/cm²   |           |
| HU           | Humidité relative moyenne quotidienne ]00UTC-00UTC]                                                                          | %       |           |
| EVAP         | Evapotranspiration totale, cumul quotidien ]06UTC-06UTC]                                                                     | mm      | 1/10      |
| ETP          | Evapotranspiration potentielle (formule de Penman-Monteith)                                                                 | mm      | 1/10      |
| PE           | Pluies efficaces, cumul quotidien ]06UTC-06UTC]                                                                              | mm      | 1/10      |
| SWI          | Indice d'humidité des sols, moyenne quotidienne [06UTC-06UTC]                                                                | %       |           |
| SSWI_10J     | Indice de sécheresse de l’humidité des sols intégré sur 10 jours (jour J et 9 jours précédents)                             | -       |           |
| DRAINC       | Drainage, cumul quotidien ]06UTC-06UTC]                                                                                      | mm      | 1/10      |
| RUNC         | Ruissellement, cumul quotidien ]06UTC-06UTC]                                                                                | mm      | 1/10      |
| RESR_NEIGE   | Équivalent en eau du manteau neigeux, moyenne quotidienne [06UTC-06UTC]                                                      | mm      | 1/10      |
| RESR_NEIGE6  | Équivalent en eau du manteau neigeux à 06 UTC                                                                               | mm      | 1/10      |
| HTEURNEIGE   | Épaisseur du manteau neigeux, moyenne quotidienne [06UTC-06UTC]                                                              | m       |           |
| HTEURNEIGE6  | Épaisseur du manteau neigeux à 06 UTC                                                                                       | m       |           |
| HTEURNEIGEX  | Épaisseur du manteau neigeux horaire maximum au cours de la journée                                                         | m       |           |
| SNOW_FRAC    | Fraction de maille recouverte par la neige, moyenne quotidienne [06UTC-06UTC]                                               | %       |           |
| ECOULEMENT   | Écoulement à la base du manteau neigeux, cumul quotidien ]06UTC-06UTC]                                                      | mm      | 1/10      |
| WG_RACINE    | Contenu en eau liquide dans la couche racinaire à 06 UTC                                                                    | m³/m³   |           |
| WGI_RACINE   | Contenu en eau gelée dans la couche racinaire à 06 UTC                                                                      | m³/m³   |           |
| TINF_H       | Température minimale des 24 températures horaires, période ]18UTC-18UTC]                                                    | °C      | 1/10      |
| TSUP_H       | Température maximale des 24 températures horaires, période ]06UTC-06UTC]                                                    | °C      | 1/10      |
| LAMBX        | Coordonnée du point de grille en Lambert II étendu (hectomètres)                                                            | hm      |           |
| LAMBY        | Coordonnée du point de grille en Lambert II étendu (hectomètres)                                                            | hm      |           |


In [14]:
from pathlib import Path
import pandas as pd

# Répertoire contenant les fichiers CSV
csv_dir = Path("data/raw/meteo_france")
output_file = Path("data/intermediate/meteo_france/meteo_france_combined.csv")

# Liste des fichiers CSV dans le répertoire
csv_files = list(csv_dir.glob("*.csv"))

# Vérifier s'il y a des fichiers CSV
if not csv_files:
    raise FileNotFoundError(f"Aucun fichier CSV trouvé dans le répertoire {csv_dir.resolve()}")

# Lire et concaténer tous les fichiers CSV
combined_df = pd.concat((pd.read_csv(file, sep=';') for file in csv_files), ignore_index=True)

# Sauvegarder le fichier combiné
combined_df.to_csv(output_file, index=False)

print(f"Fichiers combinés sauvegardés dans {output_file.resolve()}")

Fichiers combinés sauvegardés dans C:\Users\thoma\Desktop\code\MS_actuariat_climate_risk\Climate_Risk_Project_SWI_MS_A\data\intermediate\meteo_france\meteo_france_combined.csv


In [15]:
meteofrance_df = pd.read_csv("data/intermediate/meteo_france/meteo_france_combined.csv")

In [16]:
meteofrance_df

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS
0,2,641374,7106309,196001,"0,863"
1,2,641374,7106309,196002,"0,876"
2,2,641374,7106309,196003,"0,856"
3,2,641374,7106309,196004,"0,757"
4,2,641374,7106309,196005,"0,673"
...,...,...,...,...,...
7005175,9892,1215772,6046242,202408,"-0,019"
7005176,9892,1215772,6046242,202409,"0,007"
7005177,9892,1215772,6046242,202410,"0,17"
7005178,9892,1215772,6046242,202411,"0,126"


In [17]:
meteofrance_df['DATE'] = pd.to_datetime(meteofrance_df['DATE'], format='%Y%m')

In [18]:


# Chemin vers le fichier GeoPackage
gpkg_path = Path("data/intermediate/meteo_france/meteo_france_combined.gpkg")

# Vérifier si le fichier existe
if not gpkg_path.exists():
    # Créer le GeoDataFrame et sauvegarder dans un fichier
    gdf_metF = gpd.GeoDataFrame(
        meteofrance_df,
        geometry=[Point(xy) for xy in zip(meteofrance_df["LAMBX"], meteofrance_df["LAMBY"])],
        crs="EPSG:2154"  # Lambert 93
    )
    gdf_metF.to_file(gpkg_path, driver="GPKG")
    print(f"Fichier créé et sauvegardé à {gpkg_path.resolve()}")
else:
    print(f"Le fichier existe déjà à {gpkg_path.resolve()}")

Le fichier existe déjà à C:\Users\thoma\Desktop\code\MS_actuariat_climate_risk\Climate_Risk_Project_SWI_MS_A\data\intermediate\meteo_france\meteo_france_combined.gpkg


In [19]:
gdf_metF=gpd.read_file("data/intermediate/meteo_france/meteo_france_combined.gpkg")

In [20]:
gdf_metF

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,geometry
0,2,641374,7106309,1960-01-01,"0,863",POINT (641374 7106309)
1,2,641374,7106309,1960-02-01,"0,876",POINT (641374 7106309)
2,2,641374,7106309,1960-03-01,"0,856",POINT (641374 7106309)
3,2,641374,7106309,1960-04-01,"0,757",POINT (641374 7106309)
4,2,641374,7106309,1960-05-01,"0,673",POINT (641374 7106309)
...,...,...,...,...,...,...
7005175,9892,1215772,6046242,2024-08-01,"-0,019",POINT (1215772 6046242)
7005176,9892,1215772,6046242,2024-09-01,"0,007",POINT (1215772 6046242)
7005177,9892,1215772,6046242,2024-10-01,"0,17",POINT (1215772 6046242)
7005178,9892,1215772,6046242,2024-11-01,"0,126",POINT (1215772 6046242)


In [8]:
#gdf_202001 = gdf_metF[gdf_metF["DATE"] == 202001]
#gdf_202001.explore()

In [2]:
argile_gdf = gpd.read_file("data/raw/argile/ExpoArgile_Fxx_4326.gpkg")
argile_gdf.drop(columns=['mvt_id'], inplace=True)

In [3]:
argile_gdf

,ALEA,DPT,NIVEAU,geometry
0,Faible,29,1.0,"POLYGON ((-571213.912 6183443.57, -571137.475 ..."
1,Faible,29,1.0,"POLYGON ((-571331.852 6183457.006, -571268.951..."
2,Faible,29,1.0,"POLYGON ((-570372.508 6183436.703, -570373.105..."
3,Faible,29,1.0,"POLYGON ((-570523.889 6183927.87, -570525.083 ..."
4,Faible,29,1.0,"POLYGON ((-568697.462 6183975.643, -568710.898..."
...,...,...,...,...
1180531,Faible,2B,1.0,"POLYGON ((1062848.219 5208992.086, 1062904.651..."
1180532,Faible,2B,1.0,"POLYGON ((1064008.211 5201385.706, 1064003.434..."
1180533,Faible,2B,1.0,"POLYGON ((1064172.133 5202595.86, 1064169.147 ..."
1180534,Faible,2B,1.0,"POLYGON ((1064070.615 5203830.497, 1064084.648..."


In [4]:

def encode_dpt(val):
    if val == "2A":
        return 201
    if val == "2B":
        return 202
    try:
        return int(val)
    except ValueError:
        return None

argile_gdf["DPT"] = argile_gdf["DPT"].apply(encode_dpt).astype("Int64")

In [5]:
argile_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1180536 entries, 0 to 1180535
Data columns (total 4 columns):
 #   Column    Non-Null Count    Dtype   
---  ------    --------------    -----   
 0   ALEA      1180536 non-null  object  
 1   DPT       1180536 non-null  Int64   
 2   NIVEAU    1180536 non-null  float64 
 3   geometry  1180536 non-null  geometry
dtypes: Int64(1), float64(1), geometry(1), object(1)
memory usage: 37.2+ MB


In [6]:
argile_gdf

,ALEA,DPT,NIVEAU,geometry
0,Faible,29,1.0,"POLYGON ((-571213.912 6183443.57, -571137.475 ..."
1,Faible,29,1.0,"POLYGON ((-571331.852 6183457.006, -571268.951..."
2,Faible,29,1.0,"POLYGON ((-570372.508 6183436.703, -570373.105..."
3,Faible,29,1.0,"POLYGON ((-570523.889 6183927.87, -570525.083 ..."
4,Faible,29,1.0,"POLYGON ((-568697.462 6183975.643, -568710.898..."
...,...,...,...,...
1180531,Faible,202,1.0,"POLYGON ((1062848.219 5208992.086, 1062904.651..."
1180532,Faible,202,1.0,"POLYGON ((1064008.211 5201385.706, 1064003.434..."
1180533,Faible,202,1.0,"POLYGON ((1064172.133 5202595.86, 1064169.147 ..."
1180534,Faible,202,1.0,"POLYGON ((1064070.615 5203830.497, 1064084.648..."


In [26]:
argile_gdf.describe()

,DPT,NIVEAU
count,1180536.0,1.180536e+06
mean,46.356604,1.788817e+00
std,28.765852,6.937577e-01
min,1.0,1.000000e+00
25%,23.0,1.000000e+00
50%,46.0,2.000000e+00
75%,67.0,2.000000e+00
max,202.0,3.000000e+00


In [28]:
argile_gdf.isnull().sum()

ALEA        0
DPT         0
NIVEAU      0
geometry    0
dtype: int64

In [7]:
out_path = Path("data/intermediate/argile/argile.gpkg")
out_path.parent.mkdir(parents=True, exist_ok=True)

if not out_path.exists():
    argile_gdf.to_file(out_path, driver="GPKG")
    print(f"Fichier créé et sauvegardé à {out_path.resolve()}")
else:
    print(f"Le fichier existe déjà à {out_path.resolve()}")

Fichier créé et sauvegardé à C:\Users\thoma\OneDrive\Bureau\boulot\climate risk\Climate_Risk_Project_SWI_MS_A\data\intermediate\argile\argile.gpkg




    mvt_id : identifiant interne (souvent vide ici).

    ALEA : catégorie d’aléa RGA (Faible, Moyen, Fort, etc.) → c’est ton indicateur principal.

    DPT : code département (ici 29 pour le Finistère sur la capture).

    NIVEAU : niveau / version de la donnée (ici 1.0).

    geometry : polygones en coordonnées projetées (Lambert ou WebMercator selon le , EPSG:3857).


In [ ]:
### Importation des données ###


# Données communes
fp = Path("data") / "raw" / "communes" / "ADE_4-0_GPKG_WGS84G_FRA-ED2025-11-20.gpkg"
if not fp.exists():
	raise FileNotFoundError(f"File not found: {fp.resolve()}")
communes = gpd.read_file(fp, layer='commune')

# Données SWI quotidien regroupées en SWI mensuel
fp = Path("data") / "intermediate" / "datagouv" / "swimonthly_combined_1958to2025.gpkg"
if not fp.exists():
	raise FileNotFoundError(f"File not found: {fp.resolve()}")
swi_monthly = gpd.read_file(fp)

# Données SWI uniforme
fp = Path("data") / "intermediate" / "meteo_france" / "meteo_france_combined.gpkg"
if not fp.exists():
    raise FileNotFoundError(f"File not found: {fp.resolve()}")
meteo_france = gpd.read_file(fp)

# Données d'argile
fp = Path("data") / "intermediate" / "argile" / "argile.gpkg"
if not fp.exists():
    raise FileNotFoundError(f"File not found: {fp.resolve()}")
argile = gpd.read_file(fp)

In [ ]:
### Modification de la date sur les données SWI mensuel ###


# Modification de la date pour le premier du mois
swi_monthly['DATE'] = pd.to_datetime(swi_monthly['DATE']) - pd.offsets.MonthBegin(1)

# Filtrage des données à partir de 1960
swi_monthly = swi_monthly[swi_monthly['DATE'] >= '1960-01-01']

# Affichage de la base
display(swi_monthly)

In [ ]:
### Conversion des données SWI uniforme en float ###

# Conversion
meteo_france['SWI_UNIF_MENS'] = meteo_france['SWI_UNIF_MENS'].str.replace(',', '.').astype(np.float64)

# Affichage de la base
display(meteo_france)

In [ ]:
### Harmonisation des systèmes de coordonnées entre l'argile et le SWI mensuel ###


# Affichage du système de coordonnées de la base argile
print(argile.crs)

# Isolement des variables d'intérêt dans la base argile
cols_argile = ["geometry", "ALEA", "NIVEAU", "DPT"]

# Conversion du système de coordonnées de la base argile pour qu'il corresponde à celui des données SWI mensuel
argile=argile.to_crs(swi_monthly.crs)

In [ ]:
### Harmonisation des systèmes de coordonnées entre le SWI uniforme et le SWI mensuel ###

# Affichage du système de coordonnées de la base SWI uniforme
print(meteo_france.crs)

# Isolement des variables d'intérêt dans la base SWI uniforme
cols_meteo = ["DATE", "SWI_UNIF_MENS"]

# Conversion du système de coordonnées de la base SWI uniforme pour qu'il corresponde à celui des données SWI mensuel
meteo_france=meteo_france.to_crs(swi_monthly.crs)

# Vérification des systèmes de coordonnées
print(swi_monthly.crs)
print(meteo_france.crs)

In [ ]:
### Jointure des bases meteo_france et swi_monthly ###


# Chemin de destination de la jointure
output_fp = Path("data") / "processed" / "jointure_meteo_swimonthly_full.gpkg"
output_fp.parent.mkdir(parents=True, exist_ok=True)


# Check si la jointure existe déjà
if output_fp.exists():
    print(" Jointure déjà existante — chargement depuis le disque.")
    gdf_join = gpd.read_file(output_fp)


# Jointure
else:
    print(" Jointure inexistante — calcul de la jointure spatio-temporelle.")

    # Initialisation de la liste des résultats
    res = []

    # Jointure spatio-temporelle pour chaque date
    for d, meteo_d in meteo_france.groupby("DATE"):
        swi_d = swi_monthly[swi_monthly["DATE"] == d]
        if swi_d.empty:
            continue

        tmp = gpd.sjoin_nearest(
            meteo_d,
            swi_d.drop(columns=["DATE"]),   # évite DATE_left / DATE_right
            how="left",
            max_distance=100,               # à adapter à la maille
        )

        res.append(tmp)

    # Création du GeoDataFrame final en concaténant la liste de résultats
    gdf_join = gpd.GeoDataFrame(
        pd.concat(res, ignore_index = True),
        crs = meteo_france.crs
    )

    # Suppression des doublons (même NUMERO et DATE) en gardant la première occurrence (la plus proche)
    gdf_join = (
        gdf_join
        .sort_values("DATE")
        .drop_duplicates(subset = ["NUMERO", "DATE"], keep = "first")
    )

    # Suppression des colonnes inutiles
    cols_to_drop = [
        "LAMBX_right",
        "LAMBY_right",
        "geometry_right",
        "index_right",
    ]
    
    gdf_join = gdf_join.drop(
        columns=[c for c in cols_to_drop if c in gdf_join.columns]
    )

    # Renommage des colonnes LAMBX et LAMBY
    gdf_join = gdf_join.rename(columns={
        "LAMBX_left": "LAMBX",
        "LAMBY_left": "LAMBY"
    })

    # Sauvegarde de la jointure
    gdf_join.to_file(output_fp, driver="GPKG")
    print(f" Jointure sauvegardée : {output_fp}")


# Affichage de la jointure
display(gdf_join)

In [ ]:


output_fp = Path("data") / "processed" / "jointure_meteo_swi_argile_nearest.gpkg"
output_fp.parent.mkdir(parents=True, exist_ok=True)

cols_argile = ["geometry", "ALEA", "NIVEAU", "DPT"]


if output_fp.exists():
    print(" Jointure MF + SWI + Argile déjà existante — chargement.")
    gdf_join_argile = gpd.read_file(output_fp)
    

# -------------------------------------------------
# 2) Sinon → calcul
# -------------------------------------------------

else:
    print(" Calcul de la jointure MF + SWI + Argile (nearest, par année).")

    # CRS
    if argile.crs != gdf_join.crs:
        argile = argile.to_crs(gdf_join.crs)

    # Ajout colonne année (sécurité)
    gdf_join = gdf_join.copy()
    gdf_join["YEAR"] = gdf_join["DATE"].dt.year

    res = []

    for y, gdf_y in gdf_join.groupby("YEAR"):
        print(f"  → année {y}")

        tmp = gpd.sjoin_nearest(
            gdf_y,
            argile[cols_argile],
            how="left",
            distance_col="dist_to_argile"
        )

        # sécurité ex æquo
        tmp = (
            tmp
            .sort_values("dist_to_argile")
            .drop_duplicates(subset=["NUMERO", "DATE"], keep="first")
        )

        res.append(tmp)

    gdf_join_argile = gpd.GeoDataFrame(
        pd.concat(res, ignore_index=True),
        crs=gdf_join.crs
    )

    # Nettoyage
    gdf_join_argile = gdf_join_argile.drop(
        columns=[c for c in ["index_right"] if c in gdf_join_argile.columns]
    )
    gdf_join_argile.loc[gdf_join_argile['dist_to_argile'] > 1000, 'NIVEAU'] = 0   #on met 0 au-delà de 1km d'une zone argileuse
    gdf_join_argile.loc[gdf_join_argile['dist_to_argile'] > 1000, 'ALEA'] = 'Aucun'
    

    # -------------------------------------------------
    # 3) Sauvegarde
    # -------------------------------------------------
    gdf_join_argile.to_file(output_fp, driver="GPKG")
    print(f" Sauvegarde terminée : {output_fp}")

# -------------------------------------------------
# 4) Résultat
# -------------------------------------------------

display(gdf_join_argile)